In [1]:
import requests
import torch
import time
import psutil
import subprocess
from unidecode import unidecode
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, hamming_loss

import pandas as pd

from datasets import load_dataset

In [2]:
ds = load_dataset("higopires/RePro-categories-multilabel")

test = ds['test'].to_pandas()

test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1007 entries, 0 to 1006
Data columns (total 7 columns):
 #   Column                  Non-Null Count  Dtype 
---  ------                  --------------  ----- 
 0   review_text             1007 non-null   object
 1   ENTREGA                 1007 non-null   int64 
 2   OUTROS                  1007 non-null   int64 
 3   PRODUTO                 1007 non-null   int64 
 4   CONDICOESDERECEBIMENTO  1007 non-null   int64 
 5   INADEQUADA              1007 non-null   int64 
 6   ANUNCIO                 1007 non-null   int64 
dtypes: int64(6), object(1)
memory usage: 55.2+ KB


In [3]:
test = test[test['INADEQUADA'] == 0].reset_index(drop=True)

test = test.drop(columns=['INADEQUADA'])

test.rename(columns={'review_text': 'text'}, inplace=True)

test

,text,ENTREGA,OUTROS,PRODUTO,CONDICOESDERECEBIMENTO,ANUNCIO
0,"ESSE PRODUTO PODE ATÉ SER BOM, PORÉM, A AMERIC...",1,1,0,0,0
1,Recomendo!Os aparelhos da motorola são muito b...,0,0,1,0,0
2,"Eu ameiii o produto, pena que veio com o espe...",0,0,1,1,0
3,bom..............................................,0,0,1,0,0
4,Quero saber quando chegará mais pois gostaria ...,0,1,0,0,0
...,...,...,...,...,...,...
961,Produto veio com peças totalmente inferiores a...,0,0,0,1,1
962,Recebi Produto diferente do anunciado - com ap...,0,0,0,1,1
963,No anúncio constava três refis. Foi entregue a...,0,0,0,1,1
964,Comprei o produto na cor que está publicado no...,0,0,0,1,1


In [4]:
test.rename(columns={'CONDICOESDERECEBIMENTO': 'CONDICOES DE RECEBIMENTO'}, inplace=True)

labels = test.columns[1:]

labels

Index(['ENTREGA', 'OUTROS', 'PRODUTO', 'CONDICOES DE RECEBIMENTO', 'ANUNCIO'], dtype='object')

In [5]:
def get_ollama_memory_usage(port=11434):
    """
    Finds the process listening on the given port using psutil
    and returns its memory usage in bytes (RSS).
    Returns None if the process isn't found or can't be accessed.
    """
    for proc in psutil.process_iter(['pid', 'name']):
        try:
            # Call proc.connections() to see if it's listening on the desired port
            for conn in proc.connections(kind='inet'):
                if conn.laddr.port == port:
                    # Found the process that listens on port=11434
                    memory_info = proc.memory_info()
                    return memory_info.rss  # in bytes
        except (psutil.AccessDenied, psutil.NoSuchProcess):
            pass
    
    # If no process was found
    return None

get_ollama_memory_usage()

C:\Users\Rafael\AppData\Local\Temp\ipykernel_29252\1313842216.py:10: DeprecationWarning: connections() is deprecated and will be removed; use net_connections() instead
  for conn in proc.connections(kind='inet'):


In [6]:
def get_gpu_memory_usage():
    """
    Returns a list of used memory (in MB) for each GPU.
    """
    # Use nvidia-smi with the --query-gpu and --format flags to get just the memory usage
    command = [
        "nvidia-smi",
        "--query-gpu=memory.used",  # You can also add memory.free, name, etc.
        "--format=csv,noheader,nounits"  # CSV output with no header or units
    ]
    try:
        output = subprocess.check_output(command)
        # Decode the output from bytes to string
        output_str = output.decode("utf-8").strip()
        # Each line corresponds to one GPU's memory usage
        usage_values = [int(x) for x in output_str.split("\n")]
        return usage_values[0]
    except subprocess.CalledProcessError as e:
        print("Error running nvidia-smi:", e)
        return []


In [7]:
def classify(text, labels):

    url = "http://localhost:11434/api/chat"

    payload = {
        "model": "gemma3",
        "messages" : [
            {"role": "system", "content": "Você é um assistente de classificação. Seu objetivo é ler o texto fornecido e classificá-lo de acordo com a tarefa e os rótulos descritos. Você é capaz de lidar com tarefas de classificação multilabel com base nas instruções do user."},
            {"role": "user", "content": f"Classifique o seguinte texto com base na tarefa: Análise de categorias de avaliações de produtos e-commerce. Responda apenas com os rótulos que melhor descrevem o texto. Se houver mais de um rótulo, os separe com vírgula. Os rótulos possíveis são: {', '.join(labels)}. Texto: {text}"}
        ],
        "stream": False,
        "options": {
            "temperature": 0
        }
    }

    start_time = time.time()
    response = requests.post(url, json=payload)
    response_time = time.time() - start_time

    vram_usage = get_gpu_memory_usage()

    ram_usage_bytes = get_ollama_memory_usage(port=11434) / (1024 * 1024)

    response = response.json()
    total_time = response['total_duration'] / 1_000_000_000
    content = unidecode(response['message']['content'].lower())

    list = []

    if 'entrega' in content:
        list.append('ENTREGA')
    if 'produtos' in content:
        list.append('PRODUTOS')
    if 'condicoes de recebimento' in content:
        list.append('CONDICOES DE RECEBIMENTO')
    if 'anuncio' in content:
        list.append('ANUNCIO')
    if 'outros' in content:
        list.append('OUTROS')

    return list, response_time, vram_usage, ram_usage_bytes, total_time

In [8]:
# apply the classify function to the test set. create one column for each output
test[['prediction', 'response_time', 'vram_usage', 'ram_usage', 'total_time']] = test['text'].apply(lambda x: classify(x, labels)).apply(pd.Series)

C:\Users\Rafael\AppData\Local\Temp\ipykernel_29252\1313842216.py:10: DeprecationWarning: connections() is deprecated and will be removed; use net_connections() instead
  for conn in proc.connections(kind='inet'):


In [9]:
for label in labels:
    test[f"{label} pred"] = test.apply(lambda row: 1 if label in row['prediction'] else 0, axis=1)

test = test.drop(columns=['prediction'])

test.to_csv('results/gemma_ZS_multilabel1.csv', index=False)
test

,text,ENTREGA,OUTROS,PRODUTO,CONDICOES DE RECEBIMENTO,ANUNCIO,response_time,vram_usage,ram_usage,total_time,ENTREGA pred,OUTROS pred,PRODUTO pred,CONDICOES DE RECEBIMENTO pred,ANUNCIO pred
0,"ESSE PRODUTO PODE ATÉ SER BOM, PORÉM, A AMERIC...",1,1,0,0,0,4.518613,5495,120.019531,2.469253,1,0,0,0,0
1,Recomendo!Os aparelhos da motorola são muito b...,0,0,1,0,0,2.192425,5503,123.808594,0.154464,0,1,0,0,0
2,"Eu ameiii o produto, pena que veio com o espe...",0,0,1,1,0,2.485750,5445,126.472656,0.434644,0,0,0,1,0
3,bom..............................................,0,0,1,0,0,2.250233,5440,126.582031,0.202196,0,0,0,0,0
4,Quero saber quando chegará mais pois gostaria ...,0,1,0,0,0,2.440800,5445,127.128906,0.389101,1,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
961,Produto veio com peças totalmente inferiores a...,0,0,0,1,1,2.431746,5225,101.835938,0.401621,0,1,0,0,0
962,Recebi Produto diferente do anunciado - com ap...,0,0,0,1,1,2.607430,5225,101.835938,0.581073,0,1,0,1,0
963,No anúncio constava três refis. Foi entregue a...,0,0,0,1,1,2.337850,5441,101.835938,0.315705,1,1,0,0,0
964,Comprei o produto na cor que está publicado no...,0,0,0,1,1,2.214269,5439,101.835938,0.161707,0,1,0,0,0


In [10]:
y_true = test[labels].values
y_pred = test[[f"{label} pred" for label in labels]].values

#calculate the accuracy of the model
accuracy = accuracy_score(y_true, y_pred)
print('Accuracy: %f' % accuracy)
f1 = f1_score(y_true, y_pred, average='weighted')
print('F1 score: %f' % f1)
precision = precision_score(y_true, y_pred, average='weighted')
print('Precision: %f' % precision)
recall = recall_score(y_true, y_pred, average='weighted')
print('Recall: %f' % recall)
hamming_loss = hamming_loss(y_true, y_pred)
print('Hamming loss: %f' % hamming_loss)

Accuracy: 0.069358
F1 score: 0.281165
Precision: 0.275478
Recall: 0.333552
Hamming loss: 0.359420


c:\Users\Rafael\Desktop\Prog\TCC\venv\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [11]:
# get average response time, vram usage and ram usage
response_time_avg = test['response_time'].mean()
vram_usage_avg = test['vram_usage'].mean()
ram_usage_avg = test['ram_usage'].mean()
total_time_avg = test['total_time'].mean()

print(f'Average response time: {response_time_avg}')
print(f'Average VRAM usage: {vram_usage_avg}')
print(f'Average RAM usage: {ram_usage_avg}')
print(f'Average total time: {total_time_avg}')

Average response time: 2.4285199600223675
Average VRAM usage: 5404.457556935818
Average RAM usage: 105.23503008540372
Average total time: 0.38578164420289857


In [12]:
# save results to txt
with open('results/gemma_ZS_multilabel1.txt', 'w') as f:
    f.write(f'Accuracy: {accuracy}\n')
    f.write(f'F1 score: {f1}\n')
    f.write(f'Precision: {precision}\n')
    f.write(f'Recall: {recall}\n')
    f.write(f'Hamming loss: {hamming_loss}\n')
    f.write(f'Average response time: {response_time_avg}\n')
    f.write(f'Average VRAM usage: {vram_usage_avg}\n')
    f.write(f'Average RAM usage: {ram_usage_avg}\n')
    f.write(f'Average total time: {total_time_avg}\n')